In [8]:
# ---- Imports ----
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("/Users/Nooraldeen.Barqawi/Library/CloudStorage/OneDrive-EY/Documents/GitHub/karaj_bi_v3/data/processed/cars_model.csv")

In [4]:
df

,brand,model,trim,year,body_type,mileage,engine_size_cc,fuel,condition,body_condition,paint,exterior_color,interior_color,city,final_price
0,Abarath,Other,Base,1984.0,Truck,1000,2999.0,Diesel,Used,Other,Other,Orange,Nardo Grey,Amman,11000.0
1,Aito,Aito 7,Base,2021.0,Pickup,50000,3999.0,Diesel,Used,Excellent With No Defects,Original Paint,Blue,Blue,Aqaba,6000.0
2,Aito,Aito 7,Base,2021.0,Pickup,50000,3999.0,Diesel,Used,Excellent With No Defects,Original Paint,Blue,Blue,Aqaba,6000.0
3,Alfa Romeo,Model,Base,2020.0,Pickup,50000,499.0,Gasoline,New,Excellent With No Defects,Original Paint,Green,Baby Blue,Madaba,5000.0
4,Audi,A1,A1 Sportback,2014.0,Coupe,170000,1999.0,Gasoline,Used,Other,Total Repaint,Black,Red,Amman,12000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12568,Yudo,Other,Base,1996.0,Pickup,100000,1999.0,Diesel,Used,Excellent With No Defects,Other,White,White,Al Karak,2500.0
12569,Zx Auto,Grandlion,Standard,2026.0,Pickup,0,2999.0,Diesel,New,Excellent With No Defects,Original Paint,Black,Red,Zarqa,27900.0
12570,Zeekr,001,Base,2023.0,Sedan,14000,0.0,Electric,Used,Excellent With No Defects,Original Paint,Blue,Grey,Amman,32500.0
12571,Zotye,Other,Base,2008.0,Suv,19000,1999.0,Gasoline,Used,Excellent With No Defects,Total Repaint,Silver,Blue,Al Karak,2300.0


In [6]:
# ---- Features / target ----
target = "final_price"
y = df[target]
X = df.drop(columns=[target])

# Cast string/object columns to 'category' so LightGBM handles them natively
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
X[cat_cols] = X[cat_cols].astype("category")

In [9]:
# ---- Train / test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
# ---- Model ----
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    random_state=42,
)

# Early stopping on validation set
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="mae",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead 

,num_leaves,63
,learning_rate,0.05
,n_estimators,2000
,random_state,42
,feature_fraction,0.9
,bagging_fraction,0.9
,bagging_freq,5
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None


In [11]:
# ---- Evaluation ----
pred = model.predict(X_test)

print(f"MAE : {mean_absolute_error(y_test, pred):,.0f} JOD")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, pred)):,.0f} JOD")
print(f"R²  : {r2_score(y_test, pred):.4f}")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
MAE : 1,806 JOD
RMSE: 4,367 JOD
R²  : 0.8607


In [12]:
# ---- Feature importance ----
imp = (
    pd.DataFrame({"feature": X.columns, "importance": model.feature_importances_})
    .sort_values("importance", ascending=False)
)
imp.head(15)

,feature,importance
5,mileage,39965
3,year,31087
6,engine_size_cc,12494
1,model,9763
11,exterior_color,6679
0,brand,6223
2,trim,5000
12,interior_color,4787
13,city,2027
4,body_type,1870


In [13]:
# ---- Accuracy / error metrics ----
pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2   = r2_score(y_test, pred)
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100  # mean abs % error

# Tolerance accuracy: % of predictions within ±X% of true price
within_10 = np.mean(np.abs((y_test - pred) / y_test) <= 0.10) * 100
within_20 = np.mean(np.abs((y_test - pred) / y_test) <= 0.20) * 100

print(f"MAE         : {mae:,.0f} JOD")
print(f"RMSE        : {rmse:,.0f} JOD")
print(f"R²          : {r2:.4f}")
print(f"MAPE        : {mape:.2f}%")
print(f"Accuracy ±10%: {within_10:.1f}%")
print(f"Accuracy ±20%: {within_20:.1f}%")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
MAE         : 1,806 JOD
RMSE        : 4,367 JOD
R²          : 0.8607
MAPE        : 38.98%
Accuracy ±10%: 55.0%
Accuracy ±20%: 72.7%


In [14]:
# ---- Save the model ----
import joblib

joblib.dump(model, "lgbm_car_price.pkl")
print("Model saved to lgbm_car_price.pkl")

Model saved to lgbm_car_price.pkl


In [16]:
import joblib

# Build a nested mapping: brand -> model -> [trims]
brand_model_trim = (
    df.groupby(["brand", "model"])["trim"]
      .unique()
      .apply(lambda x: sorted(x.tolist()))
      .to_dict()
)
# Flatten to {brand: {model: [trims]}}
nested = {}
for (b, m), trims in brand_model_trim.items():
    nested.setdefault(b, {})[m] = trims

feature_options = {
    "categorical": {col: sorted(df[col].dropna().unique().tolist()) for col in cat_cols},
    "feature_order": X.columns.tolist(),
    "cat_cols": cat_cols,
    "brand_model_trim": nested,   # NEW
}
joblib.dump(feature_options, "feature_options.pkl")

['feature_options.pkl']